[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Categorical Features


## What you will be able to do

Give LightGBM and XGBoost a column of categories as it is, read how their trees split on it, and
make sure new data reaches a model with the same categories it was trained on.

## The idea

### The problem

The station-days table describes each station's terrain with one of five words: coastal, fjord,
inland, valley or mountain. On a clear, calm night a valley collects cold air and a coastal station
barely cools, so terrain matters, and it matters most in combination with the sky and the wind. A
tree splits a column by comparing it with a threshold, and a word has no threshold.

Numbering the words, coastal as 0 up to mountain as 4, is one common way to turn them into numbers,
and it gives the model an order that means nothing, so a split at 2.5 puts fjords with
coastal stations only because of where they fell in the list. One column of 0 and 1 for each
category, as the **scikit-learn, Deep Dive** guide did with `OneHotEncoder`, has no false order, but
each split can then ask about one category at a time, valley or not, and a table with hundreds of
categories becomes hundreds of columns.

### A formal definition

> A **categorical feature** is a column whose values name groups that have no order, such as a
> station's terrain. LightGBM and XGBoost split one **natively**, with a split that sends a set of
> categories down one branch and the rest down the other, and the library searches for the set that
> lowers the loss most.

### Why it works that way

Trying every way to divide five categories into two groups would be cheap, and for a hundred it
would be impossible, so neither library does it. LightGBM sorts the categories by the average
residual of their rows in the node and then scans that order as if it were a number, which finds a
good division with one pass, and XGBoost's partition splits rest on the same idea. With few
categories both libraries ask about one category against the rest instead, and a setting,
`max_cat_to_onehot`, decides how few.

The categories travel with the model by name. A pandas category column carries its list of
categories beside the codes that stand for them, and both libraries keep the training table's list,
so a new table is matched by the names in it, whatever order its own list is in. That is also where
the mistakes happen. A column that arrives as text, or as codes, no longer looks like the column the
model learned, and a category the model never saw has no branch of its own. LightGBM sends it where
missing values go, and XGBoost raises an error.

### Where this shows up

The **Missing Values** notebook shows how each library chooses the side of a split a missing number
takes, and the **A Frost Forecast** notebook asks its saved models about a planned night, whose terrain it
builds with the training table's categories.

### What this notebook covers

- terrain as a pandas category, its list of categories and its codes
- native categories, codes and one-hot columns in both libraries, side by side
- what a split on a category looks like inside each library
- how a new table's categories are matched to the training table's
- three mistakes with categories at prediction time

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd
import lightgbm as lgb

table = pd.DataFrame({
    "terrain": pd.Categorical(["valley", "coastal", "inland"] * 100),
    "tmin": [-8.0, 1.0, -3.0] * 100,
})
model = lgb.LGBMRegressor(n_estimators=50, verbose=-1)
model.fit(table[["terrain"]], table["tmin"])

new = pd.DataFrame({"terrain": pd.Categorical(["coastal", "valley"])})
print(model.predict(new).round(1))
```

```
[ 1. -8.]
```

The training table's categories are coastal, inland and valley, so valley's code there is 2. The new
table lists only coastal and valley, and gives valley the code 1. The model matched the two by name
and predicted each terrain's own mean, 1 and −8, from a column it was given as words, with no
numbering and no extra columns.

## Setup

Two libraries for the table, the two boosting libraries, and one score.

- `numpy` and `pandas` build the table, and pandas holds `terrain` as a category
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` scores the models on the validation and test stations

`station_days` makes 20,000 rows of daily weather at 400 weather stations in Norway, the table of the
**Why Gradient Boosting** notebook. The stations and their days are made up, and a generator started
from a fixed seed draws them, so the table is the same on every run. `terrain` is a pandas category
with the five names in `TERRAINS`, and `FEATURES` names the eight columns a model is given. The cell
splits the stations into 300 for training, 50 for validation and 50 for testing, keeping all of a
station's days on one side. This notebook compares encodings on the validation stations.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### Terrain as a pandas category

In [2]:
print(train["terrain"].cat.categories.tolist())
print(train["terrain"].head(3).tolist(), "are stored as", train["terrain"].cat.codes.head(3).tolist())
print(train["terrain"].value_counts(sort=False).to_dict())

['coastal', 'fjord', 'inland', 'valley', 'mountain']
['inland', 'inland', 'inland'] are stored as [2, 2, 2]
{'coastal': 3150, 'fjord': 3350, 'inland': 3600, 'valley': 3000, 'mountain': 1900}


The five names are the column's categories, in the order `TERRAINS` gives them, and each row holds a
code, its terrain's position in that list, so inland is 2. Every terrain has between 1,900 and 3,600
training rows.

### Native categories in both libraries

Both libraries see that `terrain` is a pandas category and split it natively, with no setting.

In [3]:
y_train, y_valid, y_test = train["tmin"], valid["tmin"], test["tmin"]

lightgbm_native = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_native.fit(train[FEATURES], y_train)
xgboost_native = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=0)
xgboost_native.fit(train[FEATURES], y_train)

print("LightGBM:", round(mean_squared_error(y_test, lightgbm_native.predict(test[FEATURES])), 3))
print("XGBoost:", round(mean_squared_error(y_test, xgboost_native.predict(test[FEATURES])), 3))

LightGBM: 2.542
XGBoost: 2.618


These are the losses of the **Why Gradient Boosting** notebook's two models, 2.542 and 2.618, since
they are the same models trained on the same stations.

### Codes and one-hot columns beside native categories

The cell trains the same two models again and scores them on the validation stations, with terrain
given as its codes, 0 to 4, which both libraries read as an ordinary number, then as five one-hot
columns, and then left out.

In [4]:
WITHOUT_TERRAIN = [name for name in FEATURES if name != "terrain"]
versions = {
    "native": lambda rows: rows[FEATURES],
    "codes": lambda rows: rows[FEATURES].assign(terrain=rows["terrain"].cat.codes),
    "one-hot": lambda rows: pd.get_dummies(rows[FEATURES], columns=["terrain"], dtype=float),
    "no terrain": lambda rows: rows[WITHOUT_TERRAIN],
}
for name, make in versions.items():
    lightgbm_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
    xgboost_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=0)
    lightgbm_model.fit(make(train), y_train)
    xgboost_model.fit(make(train), y_train)
    print(f"{name:10} {make(train).shape[1]:2} columns | LightGBM",
          f"{mean_squared_error(y_valid, lightgbm_model.predict(make(valid))):.3f} | XGBoost",
          f"{mean_squared_error(y_valid, xgboost_model.predict(make(valid))):.3f}")

native      8 columns | LightGBM 2.580 | XGBoost 2.531
codes       8 columns | LightGBM 2.592 | XGBoost 2.710
one-hot    12 columns | LightGBM 2.547 | XGBoost 2.553
no terrain  7 columns | LightGBM 3.150 | XGBoost 3.305


The native row trains the models above again and scores them on the validation stations in place of
the test stations. Leaving terrain out raises the loss from 2.580 to 3.150 in LightGBM and from 2.531 to 3.305
in XGBoost, by about 0.6 and 0.8. The three ways of giving terrain are within 0.05 of each other in
LightGBM and within 0.18 in XGBoost, where codes come out highest, and the lowest differs between the
libraries, one-hot columns for LightGBM and native categories for XGBoost. These losses come from one
split of the stations, so they do not rank the encodings. With five terrains the choice of encoding
moves the loss less than leaving terrain out does, and native categories keep the table at eight
columns and let one split send several terrains the same way, which the next cells show.

### What a split on terrain looks like

`trees_to_dataframe` lists every split of every tree. In LightGBM's table a split on a category has
the decision `==`, and its threshold lists the codes of the categories that go one way, joined by
`||`.

In [5]:
lightgbm_splits = lightgbm_native.booster_.trees_to_dataframe()
on_terrain = lightgbm_splits[lightgbm_splits["split_feature"] == "terrain"]
print(on_terrain["tree_index"].nunique(), "of 300 trees split on terrain")
for threshold in on_terrain["threshold"].head(5):
    print(f"{threshold:8}", [TERRAINS[int(code)] for code in threshold.split("||")])

174 of 300 trees split on terrain
0        ['coastal']
0||4     ['coastal', 'mountain']
0||1||4  ['coastal', 'fjord', 'mountain']
0        ['coastal']
0        ['coastal']


174 of LightGBM's 300 trees split on terrain at least once. Its first split on terrain, a threshold of
`0`, sends coastal stations one way and every other terrain the other, and its third split on terrain,
`0||1||4`, puts coastal, fjord and mountain stations on one side and inland and valley stations on the
other.

In [6]:
xgboost_splits = xgboost_native.get_booster().trees_to_dataframe()
x_on_terrain = xgboost_splits[xgboost_splits["Feature"] == "terrain"]
print(x_on_terrain["Tree"].nunique(), "of 300 trees split on terrain")
for codes in x_on_terrain["Category"].head(5):
    print(codes, [TERRAINS[code] for code in codes])

262 of 300 trees split on terrain
[1, 2, 3, 4] ['fjord', 'inland', 'valley', 'mountain']
[1, 2, 3] ['fjord', 'inland', 'valley']
[2, 3] ['inland', 'valley']
[1, 2, 3, 4] ['fjord', 'inland', 'valley', 'mountain']
[1, 2, 3, 4] ['fjord', 'inland', 'valley', 'mountain']


XGBoost splits on terrain in 262 of its 300 trees, and its `Category` column lists the codes of the
categories on one side of each split. Its first split on terrain puts fjord, inland, valley and
mountain stations against coastal ones, and its third split on terrain puts inland and valley stations
against the rest.

### A single category against the rest

LightGBM's `max_cat_to_onehot` decides how few categories a column must have for LightGBM to ask
about one category at a time. The cell trains the native model again at 5 and at 6, counts the
terrain splits that send more than one terrain the same way, and scores each model on the validation
stations.

In [7]:
for setting in (5, 6):
    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1,
                              max_cat_to_onehot=setting)
    model.fit(train[FEATURES], y_train)
    splits = model.booster_.trees_to_dataframe()
    thresholds = splits.loc[splits["split_feature"] == "terrain", "threshold"]
    print(setting, thresholds.head(5).tolist(), "|", thresholds.str.contains("||", regex=False).sum(),
          "of", len(thresholds), "group terrains | validation loss",
          round(mean_squared_error(y_valid, model.predict(valid[FEATURES])), 3))

5 ['0', '0||4', '0||1||4', '0', '0'] | 172 of 270 group terrains | validation loss 2.58
6 ['0', '0', '0', '0', '3'] | 0 of 421 group terrains | validation loss 2.547


At 5 the splits still group terrains, and at 6 every terrain split puts one terrain against the rest,
so here the setting has to be larger than the five terrains. The one-against-rest model's validation
loss is 2.547, the same as LightGBM's on one-hot columns, since a split that asks about one terrain is the
split a one-hot column allows.

### Categories are matched by name

`station_day` copies the first test row once for each terrain it is given, with a category list of
its own. The second table below lists the same five terrains in reverse, so every terrain but
inland, which stays in the middle, has a different code there.

In [8]:
def station_day(terrains, categories):
    rows = pd.concat([test[FEATURES].iloc[[0]]] * len(terrains), ignore_index=True)
    rows["terrain"] = pd.Categorical(terrains, categories=categories)
    return rows

in_order = station_day(TERRAINS, TERRAINS)
reversed_list = station_day(TERRAINS, TERRAINS[::-1])
print("codes:", in_order["terrain"].cat.codes.tolist(), reversed_list["terrain"].cat.codes.tolist())
print("LightGBM:", lightgbm_native.predict(in_order).round(3), lightgbm_native.predict(reversed_list).round(3))
print("XGBoost: ", xgboost_native.predict(in_order).round(3), xgboost_native.predict(reversed_list).round(3))

codes: [0, 1, 2, 3, 4] [4, 3, 2, 1, 0]
LightGBM: [ 0.528  0.274 -0.063 -0.646  0.555] [ 0.528  0.274 -0.063 -0.646  0.555]
XGBoost:  [ 0.457  0.068  0.292 -0.468  0.001] [ 0.457  0.068  0.292 -0.468  0.001]


The second table gives four of the five terrains another code, valley 1 in place of 3, and both models make the
same prediction for each terrain from either table. Each library stored the training table's list of
names, and reads a new table's codes through that table's own list.

### A missing terrain

A station whose terrain was never recorded has `NaN` in the column, and both libraries accept it.

In [9]:
unknown = station_day([np.nan], TERRAINS)
print("LightGBM:", lightgbm_native.predict(unknown).round(3), "| XGBoost:", xgboost_native.predict(unknown).round(3))

LightGBM: [0.137] | XGBoost: [0.441]


LightGBM predicts 0.137 for a station-day whose terrain is missing, and XGBoost 0.441. Neither library gives a missing terrain a branch of its
own. At every split on terrain it goes to the side of the terrains the split does not list, so it
travels with different terrains in different trees, and its prediction matches none of the five. The
**Missing Values** notebook shows how each library chooses the side a missing number takes.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/09-categorical-features-solutions.ipynb).

**1.** Fit an XGBoost model like `xgboost_native` with `max_cat_to_onehot=6`, under a new name, and print its validation loss and the categories of its first five terrain splits as names.

In [10]:
# your code here

**2.** Fit LightGBM on the one-hot table, the `"one-hot"` entry of `versions`, under a new name, and print how many of its 300 trees split on at least one terrain column, beside the native model's count.

In [11]:
# your code here

**3.** Make a table from the first test row with `terrain` written as `"Coastal"`, with a capital, and print what `lightgbm_native` predicts for it and what `xgboost_native` does with it. Then make the column lower case with the categories in `TERRAINS`, and predict again.

In [12]:
# your code here

**4.** In a copy of the test table, give `terrain` its categories in a shuffled order with `.cat.reorder_categories(...)`, and check that LightGBM's test loss is unchanged.

In [13]:
# your code here

## Common errors

### `ValueError: train and valid dataset categorical_feature do not match.`

The test table is given to the model without its terrain column.

In [14]:
lightgbm_native.predict(test[WITHOUT_TERRAIN])

ValueError: train and valid dataset categorical_feature do not match.

The same message comes back when terrain is there and has arrived as text, as it does from a file.

In [15]:
as_text = test[FEATURES].astype({"terrain": str})
lightgbm_native.predict(as_text)

ValueError: train and valid dataset categorical_feature do not match.

LightGBM compares the number of category columns in a new table with the training table's before it
predicts. Without terrain, and with terrain as text, the test table has none where the training table
had one, so both fail that check, with a message about categories even when the real cause is a
missing column. Give the model the same columns, with terrain as a category of the training names.

In [16]:
fixed = test[FEATURES].astype({"terrain": pd.CategoricalDtype(TERRAINS)})
print("test loss:", round(mean_squared_error(y_test, lightgbm_native.predict(fixed)), 3))

test loss: 2.542


With terrain a category of the training names, the test loss is 2.542 again.

### ``XGBoostError: Found a category not in the training set for the 3th (0-based) column: `glacier` ``

A new station stands on a glacier, a terrain no training station has. XGBoost's error starts with the
time and a path inside the library, and ends with a list of memory addresses, so the cell catches it
and prints the sentence in between.

In [17]:
glacier = station_day(["glacier"], ["glacier"])
try:
    xgboost_native.predict(glacier)
except xgb.core.XGBoostError as error:
    print(type(error).__name__ + ":", str(error).split(": ", 1)[1].splitlines()[0])

XGBoostError: Found a category not in the training set for the 3th (0-based) column: `glacier`


XGBoost will not predict for a category it never saw, and it names the column by its position, so
column 3, counting from 0, is terrain, the fourth of `FEATURES`. Map the new value to a terrain the
model knows, or to `NaN`, before predicting. The next cell casts the column to the training names,
which turns the glacier into `NaN`.

In [18]:
as_missing = glacier.astype({"terrain": pd.CategoricalDtype(TERRAINS)})
print("terrain:", as_missing["terrain"].tolist(), "| XGBoost:", xgboost_native.predict(as_missing).round(3))

terrain: [nan] | XGBoost: [0.441]


Cast to the training names, the glacier becomes `NaN`, and XGBoost predicts 0.441, its prediction
for a missing terrain in the worked examples.

### No error, and an unseen terrain treated as missing

In [19]:
print("glacier:", lightgbm_native.predict(glacier).round(3))
print("missing:", lightgbm_native.predict(unknown).round(3))
print("names the model never saw:", set(glacier["terrain"].dropna()) - set(TERRAINS))

glacier: [0.137]
missing: [0.137]
names the model never saw: {'glacier'}


LightGBM predicts 0.137 for the glacier, the same as for a missing terrain, with no warning, because it
treats a category it never saw as missing, and the third task of Your turn shows a capital letter
doing the same. The last line is the check to run before predicting, the set of names in the new table
that are not in `TERRAINS`, which is empty when every name is known.

## Recap

- You can give both libraries a pandas category column and let them split it natively.
- You can read a category split in each library's table of trees, as a set of categories.
- You can compare native categories with codes and one-hot columns on the same stations.
- You can make a new table's category column match the training table's before predicting.

## What is next

The **Classification** notebook turns the lowest temperature into frost or no frost, and bands of
cold, and trains both libraries to give probabilities.

---

&#8592; **Previous:** [Missing Values](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/08-missing-values.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Classification](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/10-classification.ipynb) &#8594;
